# 02 Peak Hours

## From 01

- 27 series: NO₂ 15 stations, PM2.5 12 (mc027, mc145, mc282 NO₂ only)
- Coverage 98.5–99.9%
- Hour = hour start (shifted −1h)
- `fireworks` flag: 31 Dec 18:00 – 1 Jan 06:00
- Max NO₂ 121, max PM2.5 276 µg/m³
- Top 10 PM2.5 hours all New Year, 6 stations in the same hour
- Top 10 NO₂ hours all March, 18:00–21:00, mostly traffic
- Values are integers

In [2]:
from pathlib import Path

import pandas as pd

# Cleaned table from 01: drops and fireworks flag already applied
ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "data" / "processed" / "hourly.csv")

# CSV stores the offset as text; parse to UTC first, then convert, as in 01
df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("Europe/Berlin")

print(df.shape)
print(df["core"].value_counts())

(236180, 7)
core
no2    131288
pm2    104892
Name: count, dtype: int64


In [6]:
# Average by hour of day: all 07:00 readings of the year become one number.
# A yearly average, not single peaks — the question is whether 07:00 is reliably bad.
# mean() skips nulls, so each average uses only the hours actually measured.
mat = (
    df.groupby(["core", "station", df["datetime"].dt.hour])["value"]
    .mean()
    .rename("mean")
    .reset_index()
    .rename(columns={"datetime": "hour"})      # the hour key inherits the source column name
)

mat.head()

,core,station,hour,mean
0,no2,mc010,0,19.435616
1,no2,mc010,1,17.585165
2,no2,mc010,2,16.200549
3,no2,mc010,3,15.882192
4,no2,mc010,4,16.463014


In [7]:
# 15 NO₂ stations × 24 h = 360, 12 PM2.5 × 24 = 288.
# Every station should show 24 hours; fewer means an hour is missing all year.
# size() counts rows including nulls, so these totals are higher than the
# valid-hour counts in 01 — coverage stays at 98.5–99.9 %.
print(mat.groupby("core").size())
print(mat.groupby(["core", "station"]).size().value_counts())

core
no2    360
pm2    288
dtype: int64
24    27
Name: count, dtype: int64


In [8]:
# Top 10 % of station-hours, cut separately per pollutant so the two
# are not compared on one scale (NO₂ and PM2.5 sit at different levels).
# H1: do the top station-hours concentrate at a few stations?
for core, g in mat.groupby("core"):
    cut = g["mean"].quantile(0.90)
    top = g[g["mean"] >= cut]
    print(
        core,
        "| cutoff", round(cut, 1),
        "| cells", len(top), "of", len(g),
        "| stations", top["station"].nunique(),
        "| hours", top["hour"].nunique(),
    )

no2 | cutoff 27.4 | cells 36 of 360 | stations 5 | hours 12
pm2 | cutoff 14.2 | cells 29 of 288 | stations 4 | hours 12
